In [1]:
import os 
os.chdir('../../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

!nvidia-smi

Mon Aug 11 19:27:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
| 89%   82C    P2            412W /  450W |   14285MiB /  24564MiB |     69%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
### Config
from easydict import EasyDict

config = EasyDict()
config.backbone = 'DiT'
config.train_pt_dir = 'samplings/dit/train_4.0/dit_train_4.0_1'
config.valid_pt_dir = 'samplings/dit/eval1000_4.0/dit_eval1000_4.0_0'
config.batch_size = 10
config.CFG = 4.0
config.epochs = 10
config.val_every = 100
config.log_dir = "logs/CFG4.0/0811-7:tempered"

### Model
from backbones.dit import DiT

if config.backbone == 'DiT':
    model = DiT(trainable=True)
print(model)


### Dataset
from datasets.pt_dataset import PtDataset
from torch.utils.data import DataLoader

train_dataset = PtDataset(config.train_pt_dir)
valid_dataset = PtDataset(config.valid_pt_dir)
print('len(train_dataset) :', len(train_dataset), 'len(valid_dataset) :', len(valid_dataset))
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('done')

### Solver
import torch
from solvers.dual.dynamic.gdual_solver_log import GDual_Solver
from solvers.transforms.tempered_loglinear_transform import TemperedLogLinearTransform
from solvers.param_extractors.gap_extractor import GAP_Extractor
from torch.utils.tensorboard import SummaryWriter

noise_schedule = model.get_noise_schedule()
extractor = GAP_Extractor(hidden_dim=128, out_dim=7, input_shape=(4, 32, 32))
solver = GDual_Solver(noise_schedule, steps=5, transform=TemperedLogLinearTransform, param_extractor=extractor, exact_first=False, skip_type="time_uniform")
solver = solver.to(model.device)
optimizer = torch.optim.AdamW(solver.parameters(), lr=1e-3)
print('done')

/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  33%|███▎      | 1/3 [00:00<00:00,  2.39it/s]An error occurred while trying to fetch /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab8

len(train_dataset) : 10000 len(valid_dataset) : 1000
done
done


In [3]:
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm

def get_valid_loss(device, solver):
    solver.eval()
    losses = []
    pbar = tqdm(valid_loader)
    for batch in pbar:
        with torch.no_grad():
            noises, conds, targets = batch['noise'].to(device, non_blocking=True), batch['cond'], batch['sample'].to(device, non_blocking=True)
            model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
            pred = solver.sample(noises, model_fn)
            loss = F.mse_loss(pred, targets)
            losses.append(loss.item())
            pbar.set_postfix({'loss': loss.item()})
            
    return np.mean(losses)
    
def do_train_loop(device, epoch, writer, solver):
    solver.train()
    pbar = tqdm(train_loader)
    losses = []
    for step, batch in enumerate(pbar):
        global_step = epoch * len(train_loader) + step
        if global_step % config.val_every == 0:
            valid_loss = get_valid_loss(device, solver)
            print('step :', global_step, 'valid_loss :', valid_loss)
            writer.add_scalar("valid/loss", valid_loss, global_step)
            save_checkpoint(global_step, config.log_dir, solver, valid_loss)

        optimizer.zero_grad(set_to_none=True)
        noises, conds, targets = batch['noise'].to(device, non_blocking=True), batch['cond'], batch['sample'].to(device, non_blocking=True)
        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            pred = solver.sample(noises, model_fn)
            loss = F.mse_loss(pred, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix({'loss': loss.item()})
        
    return np.mean(losses)

def save_checkpoint(global_step, save_dir, solver, valid_loss):
    ckpt = {
        "global_step": global_step,
        "solver_state_dict": solver.state_dict(),
        "valid_loss": float(valid_loss),
        "config": dict(config),
    }
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    
    return step_path    

print('done')

done


In [4]:
import time

t0 = time.time()
valid_loss = get_valid_loss(model.device, solver)
t1 = time.time()
print('valid_loss :', valid_loss, 'time :', t1 - t0)

100%|██████████| 100/100 [00:27<00:00,  3.69it/s, loss=1.16]

valid_loss : 1.4299472498893737 time : 27.124852895736694


### Train Loop

In [5]:
writer = SummaryWriter(log_dir=config.log_dir)

for epoch in range(config.epochs):
    train_loss = do_train_loop(model.device, epoch, writer, solver)
    print('train_loss :', train_loss)

writer.close()    

100%|██████████| 100/100 [00:27<00:00,  3.59it/s, loss=1.16]


step : 0 valid_loss : 1.4299472498893737


100%|██████████| 100/100 [00:28<00:00,  3.53it/s, loss=0.161]]


step : 100 valid_loss : 0.2872897435724735


100%|██████████| 100/100 [00:28<00:00,  3.49it/s, loss=0.16]2]  


...

100%|██████████| 100/100 [01:10<00:00,  1.41it/s, loss=nan]

step : 800 valid_loss : nan
[NaNCheck][step -1] timesteps: NaN=5, Inf=0, finite[min=1.000e+00, max=1.000e+00, mean=1.000e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] alphas: NaN=5, Inf=0, finite[min=6.330e-03, max=6.330e-03, mean=6.330e-03], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] sigmas: NaN=5, Inf=0, finite[min=1.000e+00, max=1.000e+00, mean=1.000e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_alpha: NaN=5, Inf=0, finite[min=-5.062e+00, max=-5.062e+00, mean=-5.062e+00], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_sigma: NaN=5, Inf=0, finite[min=-2.003e-05, max=-2.003e-05, mean=-2.003e-05], shape=(6,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_alpha_ratio: NaN=5, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(5,), dtype=torch.float32, device=cuda:0
[NaNCheck][step -1] log_sigma_ratio: NaN=5, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(5,), dtype=tor

[NaNCheck][step 0] logy_un: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.bfloat16, device=cuda:0
[NaNCheck][step 0] logy_vc: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.bfloat16, device=cuda:0
[NaNCheck][step 0] logy_vn: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.bfloat16, device=cuda:0
[NaNCheck][step 0] y_uc: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.float32, device=cuda:0
[NaNCheck][step 0] y_un: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.float32, device=cuda:0
[NaNCheck][step 0] y_vc: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.float32, device=cuda:0
[NaNCheck][step 0] y_vn: NaN=10, Inf=0, finite[min=nan, max=nan, mean=nan], shape=(10, 1, 1, 1), dtype=torch.float32, device=cuda:0
[NaNCheck][step 0] val_uc: NaN=10, Inf=0, finite[min=nan, max=na

 80%|████████  | 800/1000 [21:05<05:16,  1.58s/it, loss=nan]


KeyboardInterrupt: 

In [ ]:
print('done')

done
